# Figuras do paper IASEAI
Cada célula gera uma figura, salva PDF + PNG nesta pasta e mostra o plot. Paleta e tipografia ficam em `paper_style.py`.

In [ ]:
%matplotlib inline
import importlib
import paper_style as ps
importlib.reload(ps)  # reexecute após editar paper_style.py

: 

## Figura 1 — volume e composição das fontes
(`4.2.py`)

In [ ]:
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D


import paper_style as ps


# ============================================================
# LOAD RESULTS
# ============================================================

LINK_RESULTS = ps.RESULTS_DIR / "link_count_analysis_dallas" / "all_results.json"
OVERLAP_RESULTS = ps.RESULTS_DIR / "source_overlap_analysis_dallas" / "all_results.json"

OUTPUT_STEM = "figure_4_2_evidentiary_pathway"

link = json.loads(LINK_RESULTS.read_text(encoding="utf-8"))
overlap = json.loads(OVERLAP_RESULTS.read_text(encoding="utf-8"))

sources = {
    row["group"]: row["mean_links"]
    for row in link["descriptive_results"]["by_group"]
}

jaccard = {
    row["group"]: row["mean_url_jaccard"]
    for row in overlap["descriptive"]["by_group"]
}

volume_tests = pd.DataFrame(
    link["statistical_tests"]["minority_vs_majority_by_dimension"]
).set_index("dimension")

composition_tests = pd.DataFrame(
    overlap["minority_vs_majority"]["url_jaccard_distance"]
).set_index("dimension")

volume_agg = link["statistical_tests"]["aggregate_minority_vs_majority"]

(composition_agg,) = [
    row for row in overlap["minority_vs_majority"]["aggregate"]
    if row["metric"] == "url_jaccard_distance"
]


# ============================================================
# ROWS
# ============================================================

rows = []

for dimension in ps.DIMENSIONS:

    volume = volume_tests.loc[dimension]
    composition = composition_tests.loc[dimension]

    minority, majority = volume["minority_group"], volume["majority_group"]

    if (minority, majority) != (composition["minority_group"], composition["majority_group"]):
        raise RuntimeError(f"Group mismatch for {dimension}.")

    rows.append({
        "label": ps.DIMENSION_LABELS[dimension],
        "sources_majority": sources[majority],
        "sources_minority": sources[minority],
        "overlap_majority": jaccard[majority],
        "overlap_minority": jaccard[minority],
        "d_sources": volume["mean_difference"],
        "sig_sources": bool(volume["significant_holm"]),
        # Overlap = 1 - Jaccard distance, so its difference flips sign.
        "d_overlap": -100 * composition["mean_difference"],
        "sig_overlap": bool(composition["significant_permutation_holm"]),
    })

rows.append({
    "label": "Overall",
    "sources_majority": volume_agg["majority_mean"],
    "sources_minority": volume_agg["minority_mean"],
    "overlap_majority": 1 - composition_agg["majority_mean"],
    "overlap_minority": 1 - composition_agg["minority_mean"],
    "d_sources": volume_agg["mean_difference"],
    "sig_sources": volume_agg["p_value"] < 0.05,
    "d_overlap": -100 * composition_agg["mean_difference"],
    "sig_overlap": composition_agg["p_permutation_raw"] < 0.05,
})

table = pd.DataFrame(rows).set_index("label")

GENERIC_SOURCES = sources["people"]


# ============================================================
# PRINT VALUES USED IN FIGURE
# ============================================================

print("\nVALUES USED IN FIGURE\n")
print(f"Generic query: {GENERIC_SOURCES:.3f} cited sources\n")
print(table.round(4).to_string())


# ============================================================
# DRAW
# ============================================================

ps.use_paper_style()

FIG_W = ps.COLUMN_WIDTH
FIG_H = 2.50

PLOT_LEFT, PLOT_W = 0.98, 1.40
PLOT_BOTTOM, PLOT_H = 0.56, 1.74

COLUMN_X = {"d_sources": 2.64, "d_overlap": 3.06}   # centres, inches

GENERIC_Y = 7.45
ROW_Y = [6, 5, 4, 3, 2, 1, -0.45]                     # dimensions, Overall
YLIM = (-1.05, 8.05)

GLYPH = 9.4

fig = plt.figure(figsize=(FIG_W, FIG_H))
ax = ps.add_axes_in(fig, PLOT_LEFT, PLOT_BOTTOM, PLOT_W, PLOT_H)

ax.set_xlim(3.6, 10.6)
ax.set_ylim(*YLIM)
ax.set_xticks([4, 6, 8, 10])
ax.set_yticks([])
ps.strip_axes(ax, keep=("bottom",))
ax.set_xlabel("Mean cited sources per answer")

for tick in [4, 6, 8, 10]:
    ax.axvline(tick, color=ps.GRID, linewidth=0.45, zorder=0)


# ------------------------------------------------------------
# Generic query: reference glyph and line
# ------------------------------------------------------------

ax.plot(
    [GENERIC_SOURCES, GENERIC_SOURCES],
    [YLIM[0], GENERIC_Y],
    color=ps.GENERIC,
    linewidth=0.7,
    zorder=1,
)

ps.overlap_glyph(ax, GENERIC_SOURCES, GENERIC_Y, 1.0, "generic", diameter=GLYPH)


# ------------------------------------------------------------
# Dimension rows: majority -> minority
# ------------------------------------------------------------

for (label, row), y in zip(table.iterrows(), ROW_Y):

    ax.plot(
        [row["sources_majority"], row["sources_minority"]],
        [y, y],
        color=ps.RULE,
        linewidth=1.3,
        solid_capstyle="butt",
        zorder=2,
    )

    ps.overlap_glyph(ax, row["sources_majority"], y, row["overlap_majority"], "majority", diameter=GLYPH)
    ps.overlap_glyph(ax, row["sources_minority"], y, row["overlap_minority"], "minority", diameter=GLYPH)


# ------------------------------------------------------------
# Row labels, separators and the difference columns
# ------------------------------------------------------------

rows_trans = ps.fig_x_data_y(fig, ax)
label_x = ps.fig_x(fig, 0.04)

fig.text(label_x, GENERIC_Y, "Generic (“people”)", transform=rows_trans,
         ha="left", va="center", fontsize=ps.FS_LABEL, color=ps.INK_SOFT)

for (label, row), y in zip(table.iterrows(), ROW_Y):

    fig.text(label_x, y, label, transform=rows_trans, ha="left", va="center",
             fontsize=ps.FS_LABEL, color=ps.INK,
             fontweight="bold" if label == "Overall" else "normal")

    for column, decimals in [("d_sources", 1), ("d_overlap", 1)]:

        significant = row["sig_sources"] if column == "d_sources" else row["sig_overlap"]

        fig.text(
            ps.fig_x(fig, COLUMN_X[column]),
            y,
            ps.format_signed(row[column], decimals),
            transform=rows_trans,
            ha="center",
            va="center",
            fontsize=ps.FS_TICK,
            fontweight="bold" if significant else "normal",
            color=ps.INK if significant else ps.MUTED,
        )

for y in [(GENERIC_Y + ROW_Y[0]) / 2, (ROW_Y[-2] + ROW_Y[-1]) / 2]:
    fig.add_artist(Line2D(
        [label_x, ps.fig_x(fig, FIG_W - 0.03)], [y, y],
        transform=rows_trans, color=ps.RULE, linewidth=0.5,
    ))

# Table header over the difference columns: a spanner and two sub-heads.
span_left = COLUMN_X["d_sources"] - 0.21
span_right = COLUMN_X["d_overlap"] + 0.21
span_centre = (span_left + span_right) / 2

fig.text(ps.fig_x(fig, span_centre), GENERIC_Y, "Focal − comparison",
         transform=ps.offset(rows_trans, fig, y_pt=4.5),
         ha="center", va="center", fontsize=ps.FS_NOTE, color=ps.INK_SOFT)

fig.add_artist(Line2D(
    [ps.fig_x(fig, span_left), ps.fig_x(fig, span_right)], [GENERIC_Y, GENERIC_Y],
    transform=ps.offset(rows_trans, fig, y_pt=-0.5), color=ps.RULE, linewidth=0.5,
))

for column, text in [("d_sources", "sources"), ("d_overlap", "overlap")]:
    fig.text(ps.fig_x(fig, COLUMN_X[column]), GENERIC_Y, text,
             transform=ps.offset(rows_trans, fig, y_pt=-5.0),
             ha="center", va="center", fontsize=ps.FS_NOTE, color=ps.INK_SOFT)


# ------------------------------------------------------------
# Key
# ------------------------------------------------------------

key = ps.add_axes_in(fig, 0.0, 0.0, FIG_W, 0.14)
key.set_xlim(0, FIG_W)
key.set_ylim(0, 1)
key.axis("off")

KEY_ITEMS = [
    (0.10, "majority", 0.0, "Majority-marked"),
    (1.00, "minority", 0.0, "Minority-marked"),
    (1.90, "generic", 0.3, "Shared with generic"),
]

for x, condition, share, text in KEY_ITEMS:
    ps.overlap_glyph(key, x, 0.5, share, condition, diameter=7.2)
    key.text(x + 0.09, 0.5, text, ha="left", va="center",
             fontsize=ps.FS_NOTE, color=ps.INK_SOFT)

ps.save(fig, OUTPUT_STEM)
plt.show()

: 

## Figura 2 — plano de mecanismos semânticos
(`4.3.py`)

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import patheffects


import paper_style as ps


# ============================================================
# LOAD RESULTS
# ============================================================

SEMANTIC_RESULTS = ps.RESULTS_DIR / "semantic_embedding_analysis_dallas" / "all_results.json"

OUTPUT_STEM = "figure_4_3_semantic_shift_anatomy"

semantic = json.loads(SEMANTIC_RESULTS.read_text(encoding="utf-8"))

if semantic["metadata"]["primary_metric"] != "dense_distance_subject_normalized":
    raise RuntimeError("Unexpected primary semantic metric.")


def tests(rows):
    return pd.DataFrame(rows).set_index("dimension").loc[ps.DIMENSIONS]


full = tests(semantic["primary_analysis"]["dimension_tests"])
lost = tests(semantic["sentence_level"]["generic_coverage_dimension_tests"])
added = tests(semantic["sentence_level"]["group_novelty_dimension_tests"])

table = pd.DataFrame({
    "lost": lost["mean_difference"],
    "lost_low": lost["bootstrap_ci_low"],
    "lost_high": lost["bootstrap_ci_high"],
    "lost_sig": lost["significant_permutation_holm"],
    "added": added["mean_difference"],
    "added_low": added["bootstrap_ci_low"],
    "added_high": added["bootstrap_ci_high"],
    "added_sig": added["significant_permutation_holm"],
    "full": full["mean_difference"],
    "full_sig": full["significant_permutation_holm"],
})

aggregates = {
    name: semantic[section][key]
    for name, (section, key) in {
        "full": ("primary_analysis", "aggregate_minority_vs_majority"),
        "lost": ("sentence_level", "aggregate_generic_coverage"),
        "added": ("sentence_level", "aggregate_group_novelty"),
    }.items()
}

if (table["full"] <= 0).any():
    raise RuntimeError("Bubble area needs positive full-answer differences.")


# ============================================================
# PRINT VALUES USED IN FIGURE
# ============================================================

print("\nVALUES USED IN FIGURE\n")
print(table.round(4).to_string())
print("\nOutcome-level aggregates (shown in caption, not plotted):")

for name, agg in aggregates.items():
    print(
        f"  {name:5s} {agg['mean_difference']:+.4f} "
        f"[{agg['bootstrap_ci_low']:.4f}, {agg['bootstrap_ci_high']:.4f}] "
        f"p = {agg['p_permutation_raw']:.2g}"
    )


# ============================================================
# DRAW
# ============================================================

ps.use_paper_style()

FIG_W = ps.COLUMN_WIDTH
FIG_H = 3.02

AX_LEFT, AX_BOTTOM = 0.62, 0.46
AX_W = FIG_W - AX_LEFT - 0.07
AX_H = FIG_H - AX_BOTTOM - 0.08

XLIM = (-0.118, 0.172)
YLIM = (-0.088, 0.172)

MAX_DIAMETER = 17.0                      # points, for the largest effect
SCALE = MAX_DIAMETER ** 2 / table["full"].max()


def area(value):
    return value * SCALE


fig = plt.figure(figsize=(FIG_W, FIG_H))
ax = ps.add_axes_in(fig, AX_LEFT, AX_BOTTOM, AX_W, AX_H)

ax.set_xlim(*XLIM)
ax.set_ylim(*YLIM)

ticks = [-0.1, -0.05, 0, 0.05, 0.1, 0.15]
ax.set_xticks([t for t in ticks if XLIM[0] <= t <= XLIM[1]])
ax.set_yticks([t for t in ticks if YLIM[0] <= t <= YLIM[1]])

for axis in (ax.xaxis, ax.yaxis):
    axis.set_major_formatter(
        plt.FuncFormatter(lambda v, _: "0" if np.isclose(v, 0) else f"{v:.2f}".replace("-", "−"))
    )

ps.strip_axes(ax, keep=("bottom", "left"))

for tick in ax.get_xticks():
    ax.axvline(tick, color=ps.RULE if np.isclose(tick, 0) else ps.GRID,
               linewidth=0.7 if np.isclose(tick, 0) else 0.45, zorder=0)

for tick in ax.get_yticks():
    ax.axhline(tick, color=ps.RULE if np.isclose(tick, 0) else ps.GRID,
               linewidth=0.7 if np.isclose(tick, 0) else 0.45, zorder=0)

ax.set_xlabel("Generic content lost (Δ focal − comparison)")
ax.set_ylabel("New content added\n(Δ focal − comparison)", linespacing=1.15)


# ------------------------------------------------------------
# Quadrants
# ------------------------------------------------------------

quadrant_style = dict(fontsize=ps.FS_NOTE, color=ps.MUTED, style="italic")

ax.text(0.985, 0.985, "loses generic + adds new", transform=ax.transAxes,
        ha="right", va="top", **quadrant_style)
ax.text(0.015, 0.985, "adds new content", transform=ax.transAxes,
        ha="left", va="top", **quadrant_style)
ax.text(0.985, 0.015, "loses generic content", transform=ax.transAxes,
        ha="right", va="bottom", **quadrant_style)


# ------------------------------------------------------------
# Crosshairs, then bubbles from largest to smallest
# ------------------------------------------------------------

# A crosshair arm is dark when that axis' contrast is Holm-significant.
ARM_SIG, ARM_NS = ps.GENERIC, "#CACED3"

for dimension, row in table.iterrows():
    ax.plot([row["lost_low"], row["lost_high"]], [row["added"]] * 2,
            color=ARM_SIG if row["lost_sig"] else ARM_NS,
            linewidth=0.75, zorder=2, solid_capstyle="butt")
    ax.plot([row["lost"]] * 2, [row["added_low"], row["added_high"]],
            color=ARM_SIG if row["added_sig"] else ARM_NS,
            linewidth=0.75, zorder=2, solid_capstyle="butt")

for dimension, row in table.sort_values("full", ascending=False).iterrows():
    significant = bool(row["full_sig"])
    ax.scatter(
        row["lost"], row["added"],
        s=area(row["full"]),
        facecolor=ps.FOCAL if significant else ps.FOCAL_TINT,
        edgecolor=ps.WHITE if significant else ps.FOCAL,
        linewidth=0.8,
        zorder=3,
    )


# ------------------------------------------------------------
# Direct labels, each placed in free space next to its bubble
# (data coordinates of the text anchor, horizontal alignment)
# ------------------------------------------------------------

LABELS = {
    "Gender Identity": (-0.037, 0.137, "left"),
    "Disability": (0.069, 0.113, "left"),
    "Sexual Orientation": (0.032, 0.083, "right"),
    "Gender": (0.007, 0.029, "right"),
    "Ethnicity": (0.124, 0.016, "left"),
    "Race": (0.120, -0.021, "left"),
}

halo = [patheffects.withStroke(linewidth=2.2, foreground=ps.WHITE)]

for dimension, (x, y, ha) in LABELS.items():
    text = ps.DIMENSION_LABELS[dimension]
    if dimension == "Sexual Orientation":
        text = "Sexual\norientation"
    ax.text(x, y, text, ha=ha, va="center", fontsize=ps.FS_TICK, color=ps.INK,
            linespacing=1.05, path_effects=halo, zorder=4)


# ------------------------------------------------------------
# Key in the empty lower-left quadrant
# ------------------------------------------------------------

key_left = XLIM[0] + 0.007

ax.text(key_left, -0.012, "Δ full-answer distance",
        ha="left", va="center", fontsize=ps.FS_NOTE, color=ps.INK_SOFT)

for x, value in [(key_left + 0.009, 0.05), (key_left + 0.032, 0.10), (key_left + 0.059, 0.15)]:
    ax.scatter(x, -0.040, s=area(value), facecolor="none", edgecolor=ps.GENERIC,
               linewidth=0.7, zorder=3)
    ax.text(x, -0.040, f"{value:.2f}".lstrip("0"),
            transform=ps.offset(ax.transData, fig, 0, -11.0),
            ha="center", va="center", fontsize=ps.FS_NOTE - 0.5, color=ps.MUTED)

for x, significant, text in [(key_left + 0.004, True, "significant"), (key_left + 0.062, False, "n.s.")]:
    ax.scatter(x, -0.075, s=30, facecolor=ps.FOCAL if significant else ps.FOCAL_TINT,
               edgecolor=ps.WHITE if significant else ps.FOCAL, linewidth=0.8, zorder=3)
    ax.text(x, -0.075, text, transform=ps.offset(ax.transData, fig, 5.0, 0),
            ha="left", va="center", fontsize=ps.FS_NOTE - 0.5, color=ps.INK_SOFT)

ps.save(fig, OUTPUT_STEM)
plt.show()

## Figura 3 — matriz de acoplamento fontes × resposta
(`4.4.py`)

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import patheffects, transforms
from matplotlib.patches import Rectangle
from scipy.stats import rankdata, spearmanr


import paper_style as ps


# ============================================================
# LOAD RESULTS
# ============================================================

EVIDENCE_DIR = ps.RESULTS_DIR / "evidence_synthesis_analysis_dallas_v2"

MECHANISM_MAP = EVIDENCE_DIR / "mechanism_map_v2.csv"
EVIDENCE_RESULTS = EVIDENCE_DIR / "all_results_v2.json"

OUTPUT_STEM = "figure_4_4_pathway_surface_coupling_v2"

SOURCE = "url_jaccard_distance"
ANSWER = "dense_distance_subject_normalized"

data = pd.read_csv(MECHANISM_MAP)
evidence = json.loads(EVIDENCE_RESULTS.read_text(encoding="utf-8"))

if len(data) != 252 or data[[SOURCE, ANSWER]].isna().any().any():
    raise RuntimeError("Expected 252 complete explicit-group conditions.")

(reported,) = [
    row for row in evidence["pipeline"]["blocked_correlations"]
    if row["metric_x"] == SOURCE and row["metric_y"] == ANSWER
]


# ============================================================
# REPRODUCE THE BLOCKED RANK CORRELATION
# ============================================================

def centered_ranks(series):
    ranks = rankdata(series, method="average")
    return ranks - ranks.mean()


blocked_r = np.corrcoef(
    data.groupby("outcome_id")[SOURCE].transform(centered_ranks),
    data.groupby("outcome_id")[ANSWER].transform(centered_ranks),
)[0, 1]

if not np.isclose(blocked_r, reported["blocked_rank_correlation"], atol=1e-10):
    raise RuntimeError(
        f"Blocked correlation mismatch: {blocked_r} vs "
        f"{reported['blocked_rank_correlation']}"
    )


# ============================================================
# MATRIX LAYOUT
# ============================================================

DOMAINS = {
    "Healthcare": "Healthcare",
    "Employment": "Employment",
    "Education": "Education",
    "Housing": "Housing",
    "Credit & Financial Services": "Credit & finance",
    "Criminal Justice": "Criminal justice",
    "Government Benefits": "Gov. benefits",
}

if set(data["domain"]) != set(DOMAINS):
    raise RuntimeError("Unexpected domains in the mechanism map.")

data["share"] = 1 - data[SOURCE]
data["position"] = data.groupby("outcome_id")[ANSWER].rank(method="first").astype(int)

DOMAIN_GAP = 0.45

outcomes = []
y = 0.0

for domain in DOMAINS:
    for outcome in sorted(data.loc[data["domain"] == domain, "outcome"].unique()):
        outcomes.append({"domain": domain, "outcome": outcome, "y": y})
        y -= 1
    y -= DOMAIN_GAP

outcomes = pd.DataFrame(outcomes)
outcomes["outcome_id"] = outcomes["domain"] + " :: " + outcomes["outcome"]

if set(outcomes["outcome_id"]) != set(data["outcome_id"]):
    raise RuntimeError("Outcome identifiers do not match the mechanism map.")


def outcome_rho(group):
    if group[SOURCE].nunique() < 2:
        return np.nan
    return spearmanr(group[SOURCE], group[ANSWER]).statistic


rho = data.groupby("outcome_id")[[SOURCE, ANSWER]].apply(outcome_rho)
outcomes["rho"] = outcomes["outcome_id"].map(rho)

by_position = data.groupby("position")["share"].apply(lambda s: (s > 0).mean())


# ============================================================
# PRINT VALUES USED IN FIGURE
# ============================================================

print("\nVALUES USED IN FIGURE\n")
print(
    f"Blocked rank correlation: r = {blocked_r:.4f} "
    f"[{reported['cluster_bootstrap_ci_low']:.4f}, "
    f"{reported['cluster_bootstrap_ci_high']:.4f}], "
    f"p_Holm = {reported['p_blocked_permutation_holm']:.4g}"
)
print("\nWithin-outcome Spearman rho:")
print(outcomes[["outcome_id", "rho"]].round(3).to_string(index=False))
print(f"\nPositive in {(outcomes['rho'] > 0).sum()} of {outcomes['rho'].notna().sum()} defined outcomes")
print("\nShare of conditions sharing >= 1 source with the generic query, by position:")
print(by_position.round(3).to_string())


# ============================================================
# DRAW
# ============================================================

ps.use_paper_style()

FIG_W = ps.TEXT_WIDTH

ROW_PITCH = 0.089                         # inches per row
Y_TOP, Y_BOTTOM = 0.55, outcomes["y"].min() - 0.55
MATRIX_H = (Y_TOP - Y_BOTTOM) * ROW_PITCH

DOMAIN_X, OUTCOME_X = 0.04, 0.98
MATRIX_LEFT, MATRIX_W = 2.42, 3.02
RHO_LEFT, RHO_W = 5.78, 1.10

BARS_BOTTOM, BARS_H = 0.20, 0.36
MATRIX_BOTTOM = BARS_BOTTOM + BARS_H + 0.10
FIG_H = MATRIX_BOTTOM + MATRIX_H + 0.50

GLYPH = 5.3
XLIM = (0.5, 12.5)

fig = plt.figure(figsize=(FIG_W, FIG_H))

ax = ps.add_axes_in(fig, MATRIX_LEFT, MATRIX_BOTTOM, MATRIX_W, MATRIX_H)
ax_rho = ps.add_axes_in(fig, RHO_LEFT, MATRIX_BOTTOM, RHO_W, MATRIX_H, sharey=ax)
ax_bars = ps.add_axes_in(fig, MATRIX_LEFT, BARS_BOTTOM, MATRIX_W, BARS_H)

ax.set_xlim(*XLIM)
ax_bars.set_xlim(*XLIM)
ax.set_ylim(Y_BOTTOM, Y_TOP)
ax.set_xticks([])
ax.set_yticks([])
ps.strip_axes(ax)


# ------------------------------------------------------------
# Domain bands and row labels
# ------------------------------------------------------------

rows_trans = ps.fig_x_data_y(fig, ax)

for index, domain in enumerate(DOMAINS):

    ys = outcomes.loc[outcomes["domain"] == domain, "y"]

    if index % 2 == 0:
        fig.add_artist(Rectangle(
            (ps.fig_x(fig, 0.0), ys.min() - 0.5),
            ps.fig_x(fig, FIG_W - 0.03),
            ys.max() - ys.min() + 1.0,
            transform=rows_trans, facecolor=ps.BAND, edgecolor="none", zorder=-10,
        ))

    fig.text(ps.fig_x(fig, DOMAIN_X), ys.mean(), DOMAINS[domain], transform=rows_trans,
             ha="left", va="center", fontsize=ps.FS_TICK, fontweight="bold", color=ps.INK)

for _, row in outcomes.iterrows():
    fig.text(ps.fig_x(fig, OUTCOME_X), row["y"], row["outcome"], transform=rows_trans,
             ha="left", va="center", fontsize=ps.FS_NOTE, color=ps.INK_SOFT)


# ------------------------------------------------------------
# Glyphs
# ------------------------------------------------------------

y_of = outcomes.set_index("outcome_id")["y"]

for _, row in data.iterrows():
    ps.overlap_glyph(
        ax, row["position"], y_of[row["outcome_id"]], row["share"],
        row["condition"], diameter=GLYPH, ring=0.75,
    )

HEADER_PT, SUBHEADER_PT = 14.0, 3.5      # baselines above the matrix top

ax.text(0.5, 1, "Answer distance from the generic answer, ranked within outcome",
        transform=ps.offset(ax.transAxes, fig, 0, HEADER_PT), ha="center", va="baseline",
        fontsize=ps.FS_NOTE, color=ps.INK_SOFT)

top_trans = ps.offset(
    transforms.blended_transform_factory(ax.transData, ax.transAxes), fig, 0, SUBHEADER_PT
)
ax.text(1, 1, "← closest", transform=top_trans, ha="center", va="baseline",
        fontsize=ps.FS_NOTE, color=ps.MUTED)
ax.text(12, 1, "farthest →", transform=top_trans, ha="center", va="baseline",
        fontsize=ps.FS_NOTE, color=ps.MUTED)


# ------------------------------------------------------------
# Within-outcome coupling
# ------------------------------------------------------------

ax_rho.set_xlim(-0.04, 1.0)
ax_rho.set_xticks([0, 0.5, 1])
ax_rho.set_xticklabels(["0", ".5", "1"])
ax_rho.tick_params(axis="y", left=False, labelleft=False)
ps.strip_axes(ax_rho, keep=("bottom",))
ax_rho.spines["bottom"].set_bounds(0, 1)

ax_rho.plot([blocked_r, blocked_r], [Y_BOTTOM + 0.25, Y_TOP], color=ps.INK_SOFT,
            linewidth=0.7, zorder=1)

rho_top = ps.offset(
    transforms.blended_transform_factory(ax_rho.transData, ax_rho.transAxes), fig, 0, SUBHEADER_PT
)
ax_rho.text(blocked_r, 1,
            f"pooled r = {blocked_r:.2f} [{reported['cluster_bootstrap_ci_low']:.2f}, "
            f"{reported['cluster_bootstrap_ci_high']:.2f}]",
            transform=rho_top, ha="center", va="baseline", fontsize=ps.FS_NOTE, color=ps.INK_SOFT)
ax_rho.text(0.5, 1, "Source–answer coupling, ρ",
            transform=ps.offset(ax_rho.transAxes, fig, 0, HEADER_PT), ha="center", va="baseline",
            fontsize=ps.FS_NOTE, color=ps.INK_SOFT)

halo = [patheffects.withStroke(linewidth=2.0, foreground=ps.WHITE)]

for _, row in outcomes.iterrows():
    if np.isnan(row["rho"]):
        ax_rho.text(0.0, row["y"], "no shared sources", ha="left", va="center",
                    fontsize=ps.FS_NOTE - 0.8, color=ps.MUTED, style="italic",
                    path_effects=halo, zorder=4)
        continue
    ax_rho.plot([0, row["rho"]], [row["y"]] * 2, color=ps.GENERIC, linewidth=0.9,
                solid_capstyle="butt", zorder=2)
    ax_rho.plot([row["rho"]], [row["y"]], marker="o", markersize=3.4, color=ps.INK,
                markeredgecolor=ps.WHITE, markeredgewidth=0.5, zorder=3)


# ------------------------------------------------------------
# Share of conditions that keep a generic source, by position
# ------------------------------------------------------------

ax_bars.bar(by_position.index, by_position.values, width=0.56, color=ps.GENERIC,
            edgecolor="none", zorder=2)
ax_bars.set_ylim(0, 0.62)
ax_bars.set_yticks([0, 0.5])
ax_bars.set_yticklabels(["0", "50%"])
ax_bars.set_xticks(range(1, 13))
ax_bars.tick_params(axis="x", length=0, pad=1.5, labelsize=ps.FS_NOTE - 0.5, labelcolor=ps.MUTED)
ps.strip_axes(ax_bars, keep=("left", "bottom"))
ax_bars.axhline(0.5, color=ps.GRID, linewidth=0.45, zorder=0)

fig.text(ps.fig_x(fig, MATRIX_LEFT - 0.30), ps.fig_y(fig, BARS_BOTTOM + BARS_H / 2),
         "Share of conditions sharing\n≥ 1 source with the generic query",
         ha="right", va="center", fontsize=ps.FS_NOTE, color=ps.INK_SOFT, linespacing=1.15)


# ------------------------------------------------------------
# Key and test, on the top line
# ------------------------------------------------------------

key = ps.add_axes_in(fig, 0.0, FIG_H - 0.17, FIG_W, 0.15)
key.set_xlim(0, FIG_W)
key.set_ylim(0, 1)
key.axis("off")

for x, condition, share, text in [
    (0.09, "majority", 0.0, "Majority-marked query"),
    (1.17, "minority", 0.0, "Minority-marked query"),
    (2.25, "generic", 0.3, "Sources shared with the generic query"),
]:
    ps.overlap_glyph(key, x, 0.45, share, condition, diameter=6.4, ring=0.8)
    key.text(x + 0.08, 0.45, text, ha="left", va="center", fontsize=ps.FS_NOTE, color=ps.INK_SOFT)

ps.save(fig, OUTPUT_STEM)
plt.show()

## Figura 4 — divergência e alinhamento
(`4.5.py`)

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde, rankdata


import paper_style as ps


# ============================================================
# LOAD RESULTS
# ============================================================

EVIDENCE_DIR = ps.RESULTS_DIR / "evidence_synthesis_analysis_dallas_v2"

EVIDENCE_RESULTS = EVIDENCE_DIR / "all_results_v2.json"
MECHANISM_MAP = EVIDENCE_DIR / "mechanism_map_v2.csv"

OUTPUT_STEM = "figure_4_5_grounding"

GAP = "evidence_semantic_gap_source_balanced"
ANSWER = "dense_distance_subject_normalized"

evidence = json.loads(EVIDENCE_RESULTS.read_text(encoding="utf-8"))
mechanism = pd.read_csv(MECHANISM_MAP)

metadata = evidence["metadata"]

if metadata["primary_metric"] != GAP:
    raise RuntimeError("Unexpected primary evidence metric.")


# ============================================================
# ELIGIBLE CONDITIONS (same rule as the pipeline)
# ============================================================

data = mechanism[
    (mechanism["fetch_coverage"] >= metadata["reference_coverage_threshold"])
    & (mechanism["n_available_sources"] >= metadata["minimum_available_sources"])
].dropna(subset=[GAP, ANSWER])

data = data[data.groupby("outcome_id")["outcome_id"].transform("size") >= 3].copy()

(reported,) = [
    row for row in evidence["pipeline"]["blocked_correlations"]
    if row["metric_x"] == GAP and row["metric_y"] == ANSWER
]


def centered_ranks(series):
    ranks = rankdata(series, method="average")
    return ranks - ranks.mean()


blocked_r = np.corrcoef(
    data.groupby("outcome_id")[GAP].transform(centered_ranks),
    data.groupby("outcome_id")[ANSWER].transform(centered_ranks),
)[0, 1]

if (
    len(data) != reported["n_observations"]
    or not np.isclose(blocked_r, reported["blocked_rank_correlation"], atol=1e-10)
):
    raise RuntimeError(
        f"Blocked correlation mismatch: {blocked_r} (n = {len(data)}) vs "
        f"{reported['blocked_rank_correlation']} (n = {reported['n_observations']})"
    )

# Alignment (higher = closer) is the negated gap: flip r and its interval.
association = {
    "r": -blocked_r,
    "ci_low": -reported["cluster_bootstrap_ci_high"],
    "ci_high": -reported["cluster_bootstrap_ci_low"],
    "p_holm": reported["p_blocked_permutation_holm"],
}


# ============================================================
# WITHIN-OUTCOME CENTRING AND TREND
# ============================================================

data["alignment"] = 1 - data[GAP]

for column, name in [(ANSWER, "x"), ("alignment", "y")]:
    data[name] = data[column] - data.groupby("outcome_id")[column].transform("mean")

X_GRID = np.linspace(-0.23, 0.24, 120)


def ols_line(frame):
    slope, intercept = np.polyfit(frame["x"], frame["y"], 1)
    return intercept + slope * X_GRID


rng = np.random.default_rng(ps.SEED)
outcome_ids = data["outcome_id"].unique()
by_outcome = {key: group for key, group in data.groupby("outcome_id")}

boot_lines = np.array([
    ols_line(pd.concat([by_outcome[key] for key in rng.choice(outcome_ids, len(outcome_ids))]))
    for _ in range(4000)
])

trend = ols_line(data)
band_low, band_high = np.percentile(boot_lines, [2.5, 97.5], axis=0)

means = data.groupby("condition")[["x", "y"]].mean()


# ============================================================
# PRINT VALUES USED IN FIGURE
# ============================================================

print("\nVALUES USED IN FIGURE\n")
print(
    f"Blocked rank correlation (answer displacement vs alignment): "
    f"r = {association['r']:.4f} [{association['ci_low']:.4f}, {association['ci_high']:.4f}], "
    f"p_Holm = {association['p_holm']:.4g}, n = {len(data)} in {data['outcome_id'].nunique()} outcomes"
)
print(f"Pearson r of centred values (display): {np.corrcoef(data['x'], data['y'])[0, 1]:.4f}")
print("\nWithin-outcome means by condition:")
print(means.round(4).to_string())


# ============================================================
# DRAW
# ============================================================

ps.use_paper_style()

FIG_W = ps.COLUMN_WIDTH
FIG_H = 3.10

MAIN_LEFT, MAIN_BOTTOM, MAIN_W, MAIN_H = 0.62, 0.50, 2.18, 2.04
MARGIN = 0.36            # thickness of the marginal panels
GAP_IN = 0.05

XLIM = (-0.245, 0.255)
YLIM = (-0.205, 0.135)

fig = plt.figure(figsize=(FIG_W, FIG_H))

ax = ps.add_axes_in(fig, MAIN_LEFT, MAIN_BOTTOM, MAIN_W, MAIN_H)
ax_top = ps.add_axes_in(fig, MAIN_LEFT, MAIN_BOTTOM + MAIN_H + GAP_IN, MAIN_W, MARGIN)
ax_right = ps.add_axes_in(fig, MAIN_LEFT + MAIN_W + GAP_IN, MAIN_BOTTOM, MARGIN, MAIN_H)

ax.set_xlim(*XLIM)
ax.set_ylim(*YLIM)
ax.set_xticks([-0.2, -0.1, 0, 0.1, 0.2])
ax.set_yticks([-0.2, -0.1, 0, 0.1])

for axis in (ax.xaxis, ax.yaxis):
    axis.set_major_formatter(
        plt.FuncFormatter(lambda v, _: "0" if np.isclose(v, 0) else f"{v:+.1f}".replace("-", "−"))
    )

ps.strip_axes(ax, keep=("bottom", "left"))

ax.axvline(0, color=ps.RULE, linewidth=0.6, zorder=0)
ax.axhline(0, color=ps.RULE, linewidth=0.6, zorder=0)

ax.set_xlabel("Answer distance from the generic answer\n(relative to the outcome mean)", linespacing=1.15)
ax.set_ylabel("Alignment with own cited evidence\n(relative to the outcome mean)", linespacing=1.15)


# ------------------------------------------------------------
# Trend band, points, trend line
# ------------------------------------------------------------

ax.fill_between(X_GRID, band_low, band_high, color=ps.GENERIC_TINT, linewidth=0, zorder=1)

for condition in ["majority", "minority"]:
    subset = data[data["condition"] == condition]
    ax.scatter(
        subset["x"], subset["y"], s=8.5,
        facecolor=ps.CONDITION[condition]["color"], edgecolor=ps.WHITE,
        linewidth=0.35, alpha=0.9, zorder=2,
    )

ax.plot(X_GRID, trend, color=ps.INK, linewidth=1.1, zorder=3)


# ------------------------------------------------------------
# Marginal densities
# ------------------------------------------------------------

def density(values, grid):
    return gaussian_kde(values)(grid)


x_grid = np.linspace(*XLIM, 300)
y_grid = np.linspace(*YLIM, 300)

for condition in ["majority", "minority"]:

    colors = ps.CONDITION[condition]
    subset = data[data["condition"] == condition]

    dx = density(subset["x"], x_grid)
    ax_top.fill_between(x_grid, dx, color=colors["tint"], alpha=0.75, linewidth=0, zorder=1)
    ax_top.plot(x_grid, dx, color=colors["color"], linewidth=0.9, zorder=2)
    ax_top.axvline(means.loc[condition, "x"], ymax=0.92, color=colors["color"], linewidth=0.8, zorder=3)

    dy = density(subset["y"], y_grid)
    ax_right.fill_betweenx(y_grid, dy, color=colors["tint"], alpha=0.75, linewidth=0, zorder=1)
    ax_right.plot(dy, y_grid, color=colors["color"], linewidth=0.9, zorder=2)
    ax_right.axhline(means.loc[condition, "y"], xmax=0.92, color=colors["color"], linewidth=0.8, zorder=3)

ax_top.set_xlim(*XLIM)
ax_top.set_ylim(0, None)
ax_right.set_ylim(*YLIM)
ax_right.set_xlim(0, None)

for marginal in (ax_top, ax_right):
    marginal.set_xticks([])
    marginal.set_yticks([])
    ps.strip_axes(marginal)


# ------------------------------------------------------------
# Key (top-left, empty under a positive association) and test
# ------------------------------------------------------------

for i, (condition, text) in enumerate([("majority", "Majority-marked"), ("minority", "Minority-marked")]):
    y = 0.965 - i * 0.075
    ax.scatter(0.035, y, s=14, transform=ax.transAxes,
               facecolor=ps.CONDITION[condition]["color"], edgecolor=ps.WHITE, linewidth=0.35, zorder=4)
    ax.text(0.035, y, text, transform=ps.offset(ax.transAxes, fig, 5.5, 0),
            ha="left", va="center", fontsize=ps.FS_NOTE, color=ps.INK_SOFT)

ax.text(
    0.98, 0.035,
    rf"$r_\mathrm{{blocked}}$ = {association['r']:.2f} "
    f"[{association['ci_low']:.2f}, {association['ci_high']:.2f}]\n"
    + ps.format_p(association["p_holm"], r"$p_\mathrm{Holm}$")
    + f", n = {len(data)}",
    transform=ax.transAxes, ha="right", va="bottom",
    fontsize=ps.FS_NOTE, color=ps.INK_SOFT, linespacing=1.3,
)

ps.save(fig, OUTPUT_STEM)
plt.show()